In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Convolutional Autoencoder — Section 5.4

## Why Convolutional Autoencoders?

A **fully-connected AE** flattens the input and treats all pixels symmetrically:

$$z = \phi(W_{\text{enc}}\,\text{vec}(x)), \qquad \hat{x} = \sigma(W_{\text{dec}}\,z)$$

This ignores spatial structure. A **Convolutional AE** instead uses conv+pool layers in the encoder and transposed-conv+upsampling layers in the decoder.

---

### Encoder (Conv + Pool)

$$h^{(l)} = \phi\!\left(\sum_{c} k^{(l,c)} * h^{(l-1,c)}\right), \quad \text{then MaxPool or AvgPool}$$

Each conv layer applies a learned kernel $k$ (e.g. $3\times 3$) via **cross-correlation**, reducing spatial dimensions via pooling.

### Decoder (Transposed Conv + Upsample)

Transposed convolution (sometimes called "deconv") is the gradient of a convolution with respect to its input. For stride $s=2$, it upsamples by $2\times$:

$$\hat{h}^{(l)} = k^{(l)\top} \star h^{(l+1)}$$

where $\star$ denotes the transposed convolution operator (insert zeros between inputs, apply conv).

---

### Key advantages of Conv-AE over FC-AE

| Property | FC-AE | Conv-AE |
|---|---|---|
| Weight sharing | No | Yes (per kernel) |
| Translation equivariance | No | Yes |
| Spatial hierarchy | No | Yes (multi-scale) |
| Parameter count (MNIST) | ~400 K | ~50 K |
| Reconstruction quality | Blurry | Sharper edges |

---

**This notebook** simulates a Conv-AE using numpy on the sklearn digits dataset (8×8 images). Since writing actual conv layers in numpy for a browser notebook is complex, we compare:
- **FC-AE**: standard fully-connected encoder/decoder
- **Conv-AE (simulated)**: uses hand-crafted 2×2 average-pooling patch features as a fixed spatial encoder, then learns a linear decoder on those features

This lets us illustrate the spatial-locality advantage without full conv backprop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from sklearn import datasets

digits = datasets.load_digits()
X = digits.data / 16.0   # shape (1797, 64), values in [0,1]
y = digits.target


# ── Architecture diagram (matplotlib patches) ─────────────────────────────────
def draw_conv_ae_arch(ax, arch_type):
    """Draw a simple architecture diagram using matplotlib patches."""
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.5, 2)
    ax.axis('off')

    if arch_type == 'fc':
        stages = [
            ('784\n(28×28)', '#2563eb'),
            ('256',          '#7c3aed'),
            ('k',            '#dc2626'),
            ('256',          '#7c3aed'),
            ('784\n(28×28)', '#059669'),
        ]
        stage_notes = ['Input', 'FC+Act', 'Latent', 'FC+Act', 'Output']
    else:
        stages = [
            ('8×8',  '#2563eb'),
            ('4×4',  '#0891b2'),
            ('k',    '#dc2626'),
            ('4×4',  '#0891b2'),
            ('8×8',  '#059669'),
        ]
        stage_notes = ['Input', 'Pool/Enc', 'Latent', 'Dec/Up', 'Output']

    x_pos = np.linspace(0.9, 9.1, len(stages))
    box_h = 1.1
    box_w = 0.9
    y_center = 0.9

    for i, ((label, color), note) in enumerate(zip(stages, stage_notes)):
        rect = mpatches.FancyBboxPatch(
            (x_pos[i] - box_w/2, y_center - box_h/2),
            box_w, box_h,
            boxstyle='round,pad=0.06',
            facecolor=color, alpha=0.85, edgecolor='white', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(
            x_pos[i], y_center, label,
            ha='center', va='center', fontsize=8,
            color='white', fontweight='bold'
        )
        ax.text(
            x_pos[i], y_center - box_h/2 - 0.18, note,
            ha='center', va='top', fontsize=7.5, color='#475569'
        )

        if i < len(stages) - 1:
            ax.annotate(
                '',
                xy=(x_pos[i+1] - box_w/2 - 0.04, y_center),
                xytext=(x_pos[i] + box_w/2 + 0.04, y_center),
                arrowprops=dict(arrowstyle='->', color='#94a3b8', lw=1.5)
            )

    mid = len(stages) // 2
    enc_xc = x_pos[:mid+1].mean()
    dec_xc = x_pos[mid:].mean()
    ax.text(enc_xc, y_center + box_h/2 + 0.25, 'Encoder →',
            ha='center', va='bottom', color='#475569', fontsize=9)
    ax.text(dec_xc, y_center + box_h/2 + 0.25, '← Decoder',
            ha='center', va='bottom', color='#475569', fontsize=9)


# ── Activation helpers ────────────────────────────────────────────────────────
def make_act(name):
    if name == 'relu':
        return (lambda x: np.maximum(0, x),
                lambda x: (x > 0).astype(float))
    if name == 'tanh':
        return (np.tanh,
                lambda x: 1 - np.tanh(x)**2)
    # leakyrelu
    return (lambda x: np.where(x > 0, x, 0.01*x),
            lambda x: np.where(x > 0, 1.0, 0.01))


# ── FC-AE training ────────────────────────────────────────────────────────────
def run_ae_numpy(latent_dim=16, n_epochs=200, arch_type='fc', activation='relu'):
    rng = np.random.default_rng(42)
    act, act_d = make_act(activation)

    if arch_type == 'conv':
        # "Conv-like" encoder: fixed 2×2 average-pool spatial features
        # Each 8×8 image → 4×4 = 16 averaged blocks (local spatial features)
        def spatial_encode(Xb):
            """Average-pool each 8x8 image into 4x4 blocks."""
            N = Xb.shape[0]
            imgs = Xb.reshape(N, 8, 8)
            pooled = np.zeros((N, 4, 4))
            for i in range(4):
                for j in range(4):
                    pooled[:, i, j] = imgs[:, 2*i:2*i+2, 2*j:2*j+2].mean(axis=(1, 2))
            return pooled.reshape(N, 16)

        Xf = spatial_encode(X)   # (N, 16) — spatial features
        n_in = 16
    else:
        Xf = X                   # (N, 64) — raw pixels
        n_in = 64

    W1 = rng.normal(0, np.sqrt(2 / n_in), (n_in, latent_dim))
    W2 = rng.normal(0, np.sqrt(2 / latent_dim), (latent_dim, 64))

    lr = 0.01
    losses = []

    for _ in range(n_epochs):
        z_pre = Xf @ W1
        z     = act(z_pre)
        xh    = z @ W2           # predict original 64-dim pixels

        loss = np.mean((X - xh) ** 2)
        losses.append(loss)

        dL   = -2 * (X - xh) / len(X)
        dW2  = z.T @ dL
        dz   = dL @ W2.T * act_d(z_pre)
        dW1  = Xf.T @ dz

        W1 -= lr * dW1
        W2 -= lr * dW2

    z_final  = act(Xf @ W1)
    xh_final = z_final @ W2

    return losses, z_final, xh_final


# ── Main visualisation ────────────────────────────────────────────────────────
def draw_conv_ae(latent_dim=16, n_epochs=200, arch_type='fc', activation='relu'):
    losses, z_final, xh = run_ae_numpy(latent_dim, n_epochs, arch_type, activation)

    fig = plt.figure(figsize=(15, 4))
    gs = fig.add_gridspec(1, 3, wspace=0.32)

    # 1. Architecture diagram
    ax_arch = fig.add_subplot(gs[0])
    draw_conv_ae_arch(ax_arch, arch_type)
    ax_arch.set_title(
        f'Architecture: {"FC-AE (flatten)" if arch_type == "fc" else "Conv-AE (simulated spatial)"}',
        fontsize=10
    )

    # 2. Loss curve
    ax_loss = fig.add_subplot(gs[1])
    ax_loss.semilogy(losses, color='#2563eb', lw=2)
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('MSE (log scale)')
    ax_loss.set_title(f'Training Loss  |  latent_dim={latent_dim}', fontsize=10)
    ax_loss.grid(True, which='both', alpha=0.3)

    # 3. Reconstruction strip (5 examples: original top, recon bottom)
    ax_recon = fig.add_subplot(gs[2])
    n_show = 5
    orig  = X[:n_show].reshape(n_show, 8, 8)
    recon = np.clip(xh[:n_show], 0, 1).reshape(n_show, 8, 8)
    strip = np.hstack([
        np.vstack([orig[i], recon[i]])
        for i in range(n_show)
    ])
    ax_recon.imshow(strip, cmap='gray_r', aspect='auto', vmin=0, vmax=1)
    ax_recon.axhline(7.5, color='#dc2626', lw=1.5, linestyle='--')
    ax_recon.set_xticks(np.arange(n_show) * 8 + 4)
    ax_recon.set_xticklabels([str(digits.target[i]) for i in range(n_show)], fontsize=10)
    ax_recon.set_yticks([])
    ax_recon.set_title('Top: original  |  Bottom: reconstruction', fontsize=10)

    arch_label = 'FC-AE' if arch_type == 'fc' else 'Conv-AE (simulated)'
    plt.suptitle(
        f'{arch_label}  |  {activation}  |  latent_dim={latent_dim}  |  final MSE={losses[-1]:.4f}',
        fontsize=10, y=0.0
    )
    plt.tight_layout()
    plt.show()


# ── Interactive widget ────────────────────────────────────────────────────────
widgets.interact(
    draw_conv_ae,
    latent_dim=widgets.IntSlider(
        min=2, max=32, step=2, value=16,
        description='Latent dim',
        continuous_update=False,
        style={'description_width': 'initial'}
    ),
    n_epochs=widgets.IntSlider(
        min=50, max=300, step=50, value=200,
        description='Epochs',
        continuous_update=False,
        style={'description_width': 'initial'}
    ),
    arch_type=widgets.Dropdown(
        options=[('FC-AE', 'fc'), ('Conv-AE (simulated)', 'conv')],
        value='fc',
        description='Architecture',
        style={'description_width': 'initial'}
    ),
    activation=widgets.Dropdown(
        options=['relu', 'tanh', 'leakyrelu'],
        value='relu',
        description='Activation',
        style={'description_width': 'initial'}
    ),
)